# 06. Zustände

In [Notebook 05](https://ktheu.github.io/bwinfN/notebooks/index.html?path=learn_bwinf1/05_breitensuche.ipynb) waren die Knoten Stationen und Felder, also Orte. Viele
Wettbewerbsaufgaben sind aber Rätsel mit **Zügen**: Wer steht wo? Wie voll ist welches Gefäß? Dann ist ein Knoten ein
**Zustand**, eine Angabe der ganzen Lage, und eine Kante ist ein Zug, der einen Zustand in den nächsten verwandelt. Die
Breitensuche bleibt genau gleich, nur die Nachbarn berechnest du jetzt mit einer Funktion `nachfolger`. Auch die Frage am
Ende von Notebook 05 (Umsteigen im U-Bahn-Netz) ist so ein Fall: Dort wäre der Zustand das Paar (Station, Linie).

In diesem Notebook lernst du:

- einen Zustand als **Tupel** aufzuschreiben (Tupel kann man in Mengen und Wörterbüchern verwenden, Listen nicht),
- die **Nachfolger** eines Zustands zu berechnen und damit die Breitensuche aus Notebook 05 zu füttern,
- die **Größe des Zustandsraums** abzuschätzen und zu wissen, wann die Suche fertig wird,
- zu erkennen, dass eine Aufgabe **nicht lösbar** ist, und das sicher zu begründen,
- mit einer **Normalform** gleichwertige Zustände zusammenzufassen,
- zwei typische Fehler zu vermeiden.

In der Wettbewerbs-Ecke füllst du Gefäße um, und zwei Figuren hüpfen gleichzeitig über eine Hüpfburg.

## 1. Wolf, Ziege und Kohl

Ein Bauer steht mit einem Wolf, einer Ziege und einem Kohlkopf am linken Ufer eines Flusses. Sein Boot trägt ihn und
höchstens **einen** der drei. Bleibt er nicht dabei, frisst der Wolf die Ziege oder die Ziege den Kohl. Wie bringt er alle
auf das rechte Ufer?

Ein Zustand sagt, wo jeder steht. Wir schreiben ein **Tupel** `(bauer, wolf, ziege, kohl)` mit `0` für das linke und `1`
für das rechte Ufer. Der Start ist `(0, 0, 0, 0)`, das Ziel `(1, 1, 1, 1)`.

Ein **Zug** ist eine Überfahrt: Der Bauer fährt allein oder mit einem, der **am selben Ufer** steht wie er. Die
Funktion `nachfolger(zustand)` gibt die Liste aller Zustände zurück, die man mit einem Zug erreicht, **ohne** dass
etwas gefressen wird. Tupel kann man nicht ändern. Man baut den neuen Zustand also als Liste und macht daraus mit
`tuple(...)` wieder ein Tupel.

#### Aufgabe 1.1

Schreibe `nachfolger(zustand)` für Wolf, Ziege und Kohl. Die Funktion `erlaubt` ist fertig.

**Erwartete Ausgabe:**
```
[(1, 0, 1, 0)]
[(0, 0, 1, 0), (0, 0, 0, 0)]
[(1, 1, 0, 1), (1, 0, 1, 1)]
```

In [ ]:
def erlaubt(zustand):
    bauer, wolf, ziege, kohl = zustand
    if wolf == ziege and bauer != wolf:      # Wolf und Ziege allein
        return False
    if ziege == kohl and bauer != ziege:     # Ziege und Kohl allein
        return False
    return True

def nachfolger(zustand):
    liste = []
    # dein Code
    return liste

print(nachfolger((0, 0, 0, 0)))
print(nachfolger((1, 0, 1, 0)))
print(nachfolger((0, 0, 0, 1)))

<details>
<summary>Tipp anzeigen</summary>

Für `i` von 0 bis 3: Der Zug ist möglich, wenn `i == 0` ist (Bauer allein) oder `zustand[i] == zustand[0]` gilt (der
Mitfahrer steht am Ufer des Bauern). Dann `neu = list(zustand)`, `neu[0]` und gegebenenfalls `neu[i]` auf das andere Ufer
setzen (`1 - ...`), `tuple(neu)` bilden und anhängen, wenn `erlaubt(neu)`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def erlaubt(zustand):
    bauer, wolf, ziege, kohl = zustand
    if wolf == ziege and bauer != wolf:      # Wolf und Ziege allein
        return False
    if ziege == kohl and bauer != ziege:     # Ziege und Kohl allein
        return False
    return True

def nachfolger(zustand):
    liste = []
    bauer = zustand[0]
    for i in range(4):                       # i = 0: Bauer allein, 1: Wolf, 2: Ziege, 3: Kohl
        if i == 0 or zustand[i] == bauer:    # mitnehmen geht nur vom Ufer des Bauern
            neu = list(zustand)
            neu[0] = 1 - bauer               # der Bauer wechselt das Ufer
            if i > 0:
                neu[i] = 1 - bauer           # und nimmt einen mit
            neu = tuple(neu)
            if erlaubt(neu):
                liste.append(neu)
    return liste

print(nachfolger((0, 0, 0, 0)))
print(nachfolger((1, 0, 1, 0)))
print(nachfolger((0, 0, 0, 1)))
```

Vom Start aus ist nur ein Zug erlaubt: die Ziege hinüberbringen. Alles andere lässt Wolf und Ziege oder Ziege und Kohl allein.
Im Zustand `(1, 0, 1, 0)` kann der Bauer allein zurück oder mit der Ziege zurück.

</details>

#### Aufgabe 1.2

Schreibe `suche(start)`: die Breitensuche mit Vorgängern aus Notebook 05, nur dass die Nachbarn von `nachfolger(zustand)`
kommen und das Ziel von `ist_ziel(zustand)` erkannt wird. Die Funktion gibt den Plan (die Liste der Zustände vom Start bis
zum Ziel) und die Zahl der entdeckten Zustände zurück, bei „nicht lösbar“ `None` statt des Plans. `suche` benutzt die
Funktionen `nachfolger` und `ist_ziel` **mit diesen Namen**. Später in diesem Notebook definierst du sie neu, dann sucht
`suche` im neuen Rätsel.

**Erwartete Ausgabe:**
```
7 Überfahrten, 10 Zustände entdeckt
Bauer Wolf Ziege Kohl | 
Wolf Kohl             | Bauer Ziege
Bauer Wolf Kohl       | Ziege
Kohl                  | Bauer Wolf Ziege
Bauer Ziege Kohl      | Wolf
Ziege                 | Bauer Wolf Kohl
Bauer Ziege           | Wolf Kohl
                      | Bauer Wolf Ziege Kohl
```

In [ ]:
from collections import deque

def ist_ziel(zustand):
    return zustand == (1, 1, 1, 1)

def suche(start):
    vorgaenger = {start: None}
    schlange = deque([start])
    # dein Code
    return None, len(vorgaenger)

def zeige_wzk(zustand):
    namen = ['Bauer', 'Wolf', 'Ziege', 'Kohl']
    links = []
    rechts = []
    for i in range(4):
        if zustand[i] == 0:
            links.append(namen[i])
        else:
            rechts.append(namen[i])
    return ' '.join(links).ljust(22) + '| ' + ' '.join(rechts)

plan, entdeckt = suche((0, 0, 0, 0))
if plan is not None:
    print(len(plan) - 1, 'Überfahrten,', entdeckt, 'Zustände entdeckt')
    for zustand in plan:
        print(zeige_wzk(zustand))

<details>
<summary>Tipp anzeigen</summary>

Genau wie `kuerzester_weg` in Notebook 05, mit `nachfolger(zustand)` statt `graph[v]`. Am Ziel (`ist_ziel(zustand)`) den Weg
über `vorgaenger` zurückverfolgen und umdrehen. Ist die Schlange leer, gib `None` zurück.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def ist_ziel(zustand):
    return zustand == (1, 1, 1, 1)

from collections import deque

def suche(start):
    vorgaenger = {start: None}
    schlange = deque([start])
    while len(schlange) > 0:
        zustand = schlange.popleft()
        if ist_ziel(zustand):
            plan = []
            while zustand is not None:           # rückwärts bis zum Start
                plan.append(zustand)
                zustand = vorgaenger[zustand]
            plan.reverse()
            return plan, len(vorgaenger)
        for neu in nachfolger(zustand):
            if neu not in vorgaenger:            # beim Einreihen markieren
                vorgaenger[neu] = zustand
                schlange.append(neu)
    return None, len(vorgaenger)                 # Schlange leer, Ziel nie gefunden

def zeige_wzk(zustand):
    namen = ['Bauer', 'Wolf', 'Ziege', 'Kohl']
    links = []
    rechts = []
    for i in range(4):
        if zustand[i] == 0:
            links.append(namen[i])
        else:
            rechts.append(namen[i])
    return ' '.join(links).ljust(22) + '| ' + ' '.join(rechts)

plan, entdeckt = suche((0, 0, 0, 0))
if plan is not None:
    print(len(plan) - 1, 'Überfahrten,', entdeckt, 'Zustände entdeckt')
    for zustand in plan:
        print(zeige_wzk(zustand))
```

Der Plan hat 7 Überfahrten. Der Bauer bringt die Ziege hinüber, fährt leer zurück, setzt Wolf (oder Kohl) über, **nimmt die
Ziege wieder mit zurück**, setzt Kohl (oder Wolf) über, fährt leer zurück und holt am Ende die Ziege. Dass man einen Schritt „rückwärts“ machen muss, findet die
Breitensuche von allein, weil sie alle Züge probiert.

</details>

#### Frage 1.3

1. Wie viele Zustände gibt es **höchstens**? Wie viele hat die Suche tatsächlich entdeckt?
2. Die Suche hat bei 1.2 aufgehört, sobald sie das Ziel herausgenommen hat. Wann würde sie bei einem Rätsel **ohne**
   Lösung aufhören, und warum darf man dann sicher sagen: „nicht lösbar“?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Jede der vier Angaben hat 2 Möglichkeiten, also gibt es höchstens 2 · 2 · 2 · 2 = **16** Zustände. Diese Zahl ist die
   **Größe des Zustandsraums**. Entdeckt hat die Suche 10 Zustände. Die anderen 6 sind Zustände, in denen etwas
   gefressen würde (der Bauer ist dann am anderen Ufer), und die erreicht man nicht.
2. Jeder Zustand kommt höchstens **einmal** in die Schlange, denn er wird beim Einreihen markiert. Es gibt nur endlich
   viele Zustände, also wird die Schlange irgendwann leer. Dann wurde **jeder erreichbare Zustand** angesehen, und
   das Ziel war nicht dabei. Das ist ein Beweis, kein Raten: „nicht lösbar“ heißt „alle 10 erreichbaren Zustände
   geprüft“. Die Laufzeit ist wie in Notebook 05 O(Zahl der Zustände · Zahl der Nachfolger je Zustand).

</details>

## 2. Ein Zahlenrätsel

Ein Zustand muss kein Tupel sein, es kann auch eine einzige Zahl sein. Das Rätsel: Starte bei **1**. In jedem Zug darfst du
entweder **3 addieren** oder die Zahl **verdoppeln**. Wie kommst du mit den wenigsten Zügen auf **50**?

Hier ist der Zustandsraum nicht von selbst endlich: Ohne Grenze könnte man immer weiter rechnen. Wir verbieten deshalb Zahlen
über 50. Das ist erlaubt, wenn man nur größer werden kann: Eine Zahl über 50 kann nie wieder auf 50 zurückfallen. Mit der
Grenze hat der Zustandsraum höchstens 50 Zustände.

#### Aufgabe 2.1

Schreibe `nachfolger(zahl)` für dieses Rätsel (nur Ergebnisse bis `OBERGRENZE`) und `ist_ziel(zahl)`. Das Ziel steht in
der Variablen `ZIEL`.

**Erwartete Ausgabe:**
```
[4, 2] [33] [43] []
```

In [ ]:
OBERGRENZE = 50
ZIEL = 50

def nachfolger(zahl):
    liste = []
    # dein Code
    return liste

def ist_ziel(zahl):
    return False

print(nachfolger(1), nachfolger(30), nachfolger(40), nachfolger(50))

<details>
<summary>Tipp anzeigen</summary>

Die beiden möglichen Ergebnisse sind `zahl + 3` und `zahl * 2`. Hänge sie nur an, wenn sie höchstens `OBERGRENZE` sind.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
OBERGRENZE = 50
ZIEL = 50

def nachfolger(zahl):
    liste = []
    for neu in [zahl + 3, zahl * 2]:
        if neu <= OBERGRENZE:                # größere Zahlen gibt es für uns nicht
            liste.append(neu)
    return liste

def ist_ziel(zahl):
    return zahl == ZIEL

print(nachfolger(1), nachfolger(30), nachfolger(40), nachfolger(50))
```

</details>

#### Aufgabe 2.2

Rufe `suche(1)` auf. Gib den Weg und die Zahl der entdeckten Zustände aus. Probiere dann auch die Ziele 49, 20 und 7.

**Erwartete Ausgabe:**
```
Ziel 50 : 6 Züge [1, 4, 8, 11, 22, 25, 50] - 34 Zustände entdeckt
Ziel 49 : 7 Züge [1, 4, 7, 10, 20, 23, 46, 49] - 34 Zustände entdeckt
Ziel 20 : 4 Züge [1, 4, 7, 10, 20] - 18 Zustände entdeckt
Ziel 7 : 2 Züge [1, 4, 7] - 6 Zustände entdeckt
```

In [ ]:
# dein Code

<details>
<summary>Tipp anzeigen</summary>

`ZIEL` neu setzen, dann `suche(1)` aufrufen. Der Plan ist die Liste der Zahlen auf dem Weg.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
for ziel in [50, 49, 20, 7]:
    ZIEL = ziel
    plan, entdeckt = suche(1)
    print('Ziel', ziel, ':', len(plan) - 1, 'Züge', plan, '-', entdeckt, 'Zustände entdeckt')
```

Für 50 braucht man 6 Züge: 1, 4, 8, 11, 22, 25, 50. Die Suche hat bis dahin 34 verschiedene Zahlen entdeckt, beim nahen Ziel 7
dagegen nur 6. Je weiter das Ziel (in Zügen) entfernt ist, desto mehr Zustände muss die Breitensuche ansehen.

</details>

#### Aufgabe 2.3

Jetzt darfst du statt zu verdoppeln **verdreifachen** (`+3` oder `·3`), und die Grenze ist 100. Kommt man von 1 auf 50?
Definiere dafür `nachfolger` neu und rufe `suche(1)` auf. Gib aus, ob es einen Plan gibt, und wie viele Zustände die Suche
entdeckt hat.

**Erwartete Ausgabe:**
```
nicht lösbar, 67 Zustände angesehen
```

In [ ]:
OBERGRENZE = 100
ZIEL = 50

# dein Code

<details>
<summary>Tipp anzeigen</summary>

Wie in 2.1, nur mit `zahl * 3`. Das Ergebnis von `suche(1)` ist bei „nicht lösbar“ ein Paar mit `None` als erstem Wert.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
OBERGRENZE = 100
ZIEL = 50

def nachfolger(zahl):
    liste = []
    for neu in [zahl + 3, zahl * 3]:
        if neu <= OBERGRENZE:
            liste.append(neu)
    return liste

plan, entdeckt = suche(1)
if plan is None:
    print('nicht lösbar,', entdeckt, 'Zustände angesehen')
else:
    print(len(plan) - 1, 'Züge')
```

</details>

#### Frage 2.4

1. Warum kommt man mit `+3` und `·3` von 1 nie auf 50, **auch ohne** Grenze? Überlege, welchen Rest die Zahlen beim
   Teilen durch 3 haben.
2. Die Suche in 2.3 hat das ohne Mathematik herausgefunden. Warum ist ihr „nicht lösbar“ trotzdem sicher? Und was geht
   schief, wenn man die Obergrenze weglässt?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. `+3` ändert den Rest beim Teilen durch 3 nicht, `·3` macht ihn zu 0. Von der 1 (Rest 1) erreicht man also nur Zahlen mit
   Rest 1 (vor dem ersten Verdreifachen) und danach nur Vielfache von 3. 50 hat Rest 2, ist also nie dabei.
2. Mit der Grenze 100 gibt es nur endlich viele Zustände, die Suche sieht alle erreichbaren an (67 Stück), und 50 ist nicht
   dabei. Die Grenze kostet keine Lösung, denn Zahlen über 50 kann man nicht wieder verkleinern, und über 100 erst recht
   nicht. **Ohne** Grenze wäre der Zustandsraum unendlich: Bei einem lösbaren Rätsel findet die Breitensuche die Lösung
   trotzdem (sie sieht alle näheren Zustände zuerst), bei einem unlösbaren läuft sie **für immer**. Die Grenze darf aber
   nie so klein sein, dass sie Lösungen abschneidet. Bei Rätseln mit „minus“ oder „geteilt durch“ muss man das genau
   überlegen.

</details>

## 3. Normalform: gleiche Zustände zusammenfassen

Auf einem Brett aus 4 × 4 Feldern liegen **zwei gleiche** Steine. Ein Zug schiebt einen Stein um ein Feld nach oben,
unten, links oder rechts, aber nicht auf das Feld des anderen Steins. Gestartet wird mit den Steinen auf `(3, 3)` und
`(3, 2)`, sie sollen auf `(0, 0)` und `(0, 1)` liegen.

Weil die Steine gleich aussehen, ist die Lage „Stein A auf `(3, 3)`, Stein B auf `(3, 2)`“ dieselbe wie „A auf `(3, 2)`,
B auf `(3, 3)`“. Als Tupel `((3, 3), (3, 2))` und `((3, 2), (3, 3))` wären das aber **zwei** Zustände, und die Suche würde
alles doppelt untersuchen. Die Abhilfe ist eine **Normalform**: Von allen gleichwertigen Schreibweisen eines Zustands
wählt man **eine** feste aus. Hier sortiert man die beiden Positionen. `tuple(sorted(zustand))` macht aus beiden
Schreibweisen `((3, 2), (3, 3))`. Wichtig: Jeder Zustand, den `nachfolger` zurückgibt, und der Start müssen in der
Normalform sein.

#### Aufgabe 3.1

Schreibe `normalform(zustand)`, die die Positionen sortiert und als Tupel zurückgibt. `nachfolger` ist fertig und ruft
`normalform` für jeden neuen Zustand auf. `zaehle_alle(start)` zählt alle Zustände, die man vom Start aus erreichen
kann. Rufe danach `suche(start)` auf und gib die Zahl der Züge und der entdeckten Zustände aus.

**Erwartete Ausgabe:**
```
((3, 2), (3, 3))
erreichbare Zustände: 120
10 Züge, 120 Zustände entdeckt
```

In [ ]:
GROESSE = 4
ZIEL_STEINE = ((0, 0), (0, 1))

def normalform(zustand):
    return zustand                           # noch nicht sortiert

def nachfolger(zustand):
    liste = []
    for i in range(2):                       # welcher Stein bewegt wird
        zeile, spalte = zustand[i]
        for dz, ds in [(0, 1), (0, -1), (1, 0), (-1, 0)]:
            neu_pos = (zeile + dz, spalte + ds)
            im_brett = 0 <= neu_pos[0] < GROESSE and 0 <= neu_pos[1] < GROESSE
            if im_brett and neu_pos != zustand[1 - i]:
                neu = list(zustand)
                neu[i] = neu_pos
                liste.append(normalform(tuple(neu)))
    return liste

def ist_ziel(zustand):
    return zustand == ZIEL_STEINE

def zaehle_alle(start):
    gesehen = {start}
    schlange = deque([start])
    while len(schlange) > 0:
        zustand = schlange.popleft()
        for neu in nachfolger(zustand):
            if neu not in gesehen:
                gesehen.add(neu)
                schlange.append(neu)
    return len(gesehen)

start = normalform(((3, 3), (3, 2)))
print(start)
print('erreichbare Zustände:', zaehle_alle(start))
# dein Code: suche(start) aufrufen

<details>
<summary>Tipp anzeigen</summary>

`sorted(zustand)` sortiert die beiden Positionen (Tupel werden zuerst nach der Zeile, dann nach der Spalte verglichen) und
gibt eine Liste zurück. Mit `tuple(...)` wird wieder ein Tupel daraus.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
GROESSE = 4
ZIEL_STEINE = ((0, 0), (0, 1))

def normalform(zustand):
    return tuple(sorted(zustand))            # Tupel von Tupeln sortiert man Position für Position

def nachfolger(zustand):
    liste = []
    for i in range(2):                       # welcher Stein bewegt wird
        zeile, spalte = zustand[i]
        for dz, ds in [(0, 1), (0, -1), (1, 0), (-1, 0)]:
            neu_pos = (zeile + dz, spalte + ds)
            im_brett = 0 <= neu_pos[0] < GROESSE and 0 <= neu_pos[1] < GROESSE
            if im_brett and neu_pos != zustand[1 - i]:
                neu = list(zustand)
                neu[i] = neu_pos
                liste.append(normalform(tuple(neu)))
    return liste

def ist_ziel(zustand):
    return zustand == ZIEL_STEINE

def zaehle_alle(start):
    gesehen = {start}
    schlange = deque([start])
    while len(schlange) > 0:
        zustand = schlange.popleft()
        for neu in nachfolger(zustand):
            if neu not in gesehen:
                gesehen.add(neu)
                schlange.append(neu)
    return len(gesehen)

start = normalform(((3, 3), (3, 2)))
print(start)
print('erreichbare Zustände:', zaehle_alle(start))
plan, entdeckt = suche(start)
print(len(plan) - 1, 'Züge,', entdeckt, 'Zustände entdeckt')
```

Mit Normalform gibt es 120 Zustände: Zwei verschiedene Felder von 16 kann man auf 16 · 15 / 2 = 120 Arten wählen. Ohne
Normalform wären es doppelt so viele, nämlich 240 (16 · 15, jede Lage zweimal). Das kannst du ausprobieren, indem
du `normalform` wieder `return zustand` zurückgeben lässt. Die Zahl der Züge bleibt gleich: Jeder Stein muss 5 Felder
weit, zusammen 10 Züge.

</details>

#### Frage 3.2

1. Drei **gleiche** Steine auf dem Brett: Um welchen Faktor schrumpft der Zustandsraum durch die Normalform ungefähr?
2. Wann darfst du Zustände **nicht** durch Sortieren zusammenfassen? Denke an Wolf, Ziege und Kohl und an zwei Figuren
   Sasha und Mika.
3. Was wäre eine Normalform für ein Brettspiel, bei dem man das Brett drehen darf?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Drei gleiche Steine kann man auf 3 · 2 · 1 = 6 Arten vertauschen. Ohne Normalform zählt jede Lage 6-mal, mit ihr nur
   einmal: Der Zustandsraum wird **6-mal** kleiner (bei k gleichen Steinen allgemein k!-mal). Der
   Gewinn wächst stark mit der Zahl der Steine.
2. Wolf, Ziege und Kohl sind **unterschiedlich**: Vertauscht man sie, ist es ein anderer Zustand (in einem frisst der Wolf
   die Ziege, im anderen nicht). Auch Sasha und Mika sind unterscheidbar. Nur wirklich gleichwertige Zustände
   darf man zusammenfassen.
3. Man dreht den Zustand auf alle vier Arten (und spiegelt ihn vielleicht) und nimmt **die kleinste** dieser Schreibweisen,
   etwa die in der Sortierung ganz vorn. Alle gleichwertigen Lagen haben dann dieselbe Normalform.

</details>

## 4. Zwei typische Fehler

#### Frage 4.1

Finde den Fehler. Lara hat `nachfolger_liste` so geschrieben, dass sie den neuen Zustand als **Liste** zurückgibt:

```python
def nachfolger_liste(zustand):
    neu = list(zustand)
    neu[0] = 1 - neu[0]                      # der Bauer fährt allein
    return [neu]

vorgaenger = {(0, 0, 0, 0): None}
for u in nachfolger_liste((0, 0, 0, 0)):
    if u not in vorgaenger:
        vorgaenger[u] = (0, 0, 0, 0)
```

Sie bekommt die Fehlermeldung `TypeError: unhashable type: 'list'`. Woran liegt das, und was muss sie ändern?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

Wörterbücher und Mengen brauchen Schlüssel, die sich **nicht ändern** können (sie müssen *hashbar* sein). Eine Liste
kann man ändern, also darf sie nicht Schlüssel in `vorgaenger` sein. Ein Tupel ist unveränderlich und geht. Lara muss den
neuen Zustand vor dem Zurückgeben mit `tuple(neu)` in ein Tupel verwandeln, am besten schon in `nachfolger`, damit
alle Zustände überall Tupel sind (auch der Start).

</details>

#### Frage 4.2

Finde den Fehler. Ben meint, es reiche, wo Wolf, Ziege und Kohl stehen, und den Bauern lässt er beim Merken weg
(`nachfolger` und `erlaubt` sind wie in 1.1):

```python
def suche_knapp(start):
    vorgaenger = {start: None}
    gesehen = {start[1:]}                    # Zustand ohne den Bauern merken
    schlange = deque([start])
    while len(schlange) > 0:
        zustand = schlange.popleft()
        if zustand == (1, 1, 1, 1):
            return 'Plan gefunden'
        for neu in nachfolger(zustand):
            if neu[1:] not in gesehen:
                gesehen.add(neu[1:])
                vorgaenger[neu] = zustand
                schlange.append(neu)
    return 'nicht lösbar'
```

`suche_knapp((0, 0, 0, 0))` gibt `nicht lösbar` zurück, obwohl das Rätsel lösbar ist. Was ist falsch?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

Der Zustand ist **zu knapp**: Nach der ersten Überfahrt ist der Zustand `(1, 0, 1, 0)`, ohne den Bauern bleibt `(0, 1, 0)`.
Fährt der Bauer allein zurück, ergibt das `(0, 0, 1, 0)`, wieder mit dem Rest `(0, 1, 0)`. Ben hält das für „schon gesehen“
und sortiert den Zug aus, obwohl der Bauer jetzt am anderen Ufer steht. Damit ist der einzige erste Zug eine Sackgasse, und
Bens Suche endet ohne Plan. Ein Zustand muss **alles** enthalten, was die möglichen Züge und das Ziel beeinflusst. Merke:
Zu viele Angaben (nicht zusammengefasste gleiche Zustände wie in Abschnitt 3) kosten nur Zeit, **zu wenige** machen das
Ergebnis falsch.

</details>

## Wettbewerbs-Ecke 1: Faires Füllen mit drei Gefäßen

*Angelehnt an Aufgabe 1 „Faires Füllen“ der 1. Runde des 33. Bundeswettbewerbs Informatik (2014/15).*

Ein Händler und ein Käufer haben ein Rätsel: Ein Gefäß mit 8 Maßen Wein ist voll, daneben stehen leere Gefäße für 5 und 3
Maße. Man darf nur von einem Gefäß in ein anderes gießen, **bis das erste leer oder das zweite voll ist**. Wie bekommt man
mit den wenigsten Schritten genau **4 Maße** in das erste Gefäß (dann liegen auch die anderen 4 in den beiden kleinen)?

Die Dateien `daten/gefaesse_*.txt` enthalten in der ersten Zeile die Fassungsvermögen, in der zweiten die Füllstände und in der
dritten die Menge, die am Ende im **ersten** Gefäß sein soll. (In der echten Aufgabe gibt es beliebig viele Gefäße und
zwei Personen, beide sollen am Ende gleich viel haben. Hier sind es drei Gefäße und ein Ziel.) Der Zustand ist das Tupel
der drei Füllstände.

#### Aufgabe E.1

Schreibe `nachfolger(zustand)` und `ist_ziel(zustand)`. Die Fassungsvermögen stehen in `GROESSE`, die Zielmenge in `ZIEL`
(beide setzt `lade` aus der Datei). Ein Zug gießt vom Gefäß `a` in ein anderes Gefäß `b`, wenn `a` nicht leer und `b` nicht
voll ist, und zwar so viel, wie hineinpasst oder wie in `a` ist (`min`).

**Erwartete Ausgabe:**
```
(8, 5, 3) (8, 0, 0) 4
[(3, 5, 0), (5, 0, 3)]
[(0, 5, 3), (8, 0, 0), (3, 2, 3)]
```

In [ ]:
def lade(dateiname):
    zeilen = open(dateiname).read().strip().split('\n')
    groesse = tuple(int(wort) for wort in zeilen[0].split())
    start = tuple(int(wort) for wort in zeilen[1].split())
    ziel = int(zeilen[2])
    return groesse, start, ziel

def nachfolger(zustand):
    liste = []
    # dein Code
    return liste

def ist_ziel(zustand):
    return False

GROESSE, start_gef, ZIEL = lade('daten/gefaesse_raetsel.txt')
print(GROESSE, start_gef, ZIEL)
print(nachfolger(start_gef))
print(nachfolger((3, 5, 0)))

<details>
<summary>Tipp anzeigen</summary>

Zwei verschachtelte Schleifen über `a` und `b` aus `range(3)`. Bedingung: `a != b`, `zustand[a] > 0` und
`zustand[b] < GROESSE[b]`. Die Menge ist `min(zustand[a], GROESSE[b] - zustand[b])`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def lade(dateiname):
    zeilen = open(dateiname).read().strip().split('\n')
    groesse = tuple(int(wort) for wort in zeilen[0].split())
    start = tuple(int(wort) for wort in zeilen[1].split())
    ziel = int(zeilen[2])
    return groesse, start, ziel

def nachfolger(zustand):
    liste = []
    for a in range(3):
        for b in range(3):
            if a != b and zustand[a] > 0 and zustand[b] < GROESSE[b]:
                menge = min(zustand[a], GROESSE[b] - zustand[b])    # so viel passt
                neu = list(zustand)
                neu[a] = neu[a] - menge
                neu[b] = neu[b] + menge
                liste.append(tuple(neu))
    return liste

def ist_ziel(zustand):
    return zustand[0] == ZIEL

GROESSE, start_gef, ZIEL = lade('daten/gefaesse_raetsel.txt')
print(GROESSE, start_gef, ZIEL)
print(nachfolger(start_gef))
print(nachfolger((3, 5, 0)))
```

Aus `(8, 0, 0)` gibt es nur zwei Züge: das mittlere Gefäß ganz füllen (`(3, 5, 0)`) oder das kleine (`(5, 0, 3)`). Aus
`(3, 5, 0)` sind es drei: vom ersten ins dritte, vom zweiten ins erste (zurück zum Start) und vom zweiten ins dritte. Das volle
zweite Gefäß kann nichts aufnehmen, und aus dem leeren dritten kann man nichts gießen.

</details>

#### Aufgabe E.2

Löse alle drei Dateien mit `suche`. Gib die Zahl der Schritte und der entdeckten Zustände aus. Hat eine Datei keine Lösung,
gib das aus. Zeige für das ursprüngliche Rätsel den Plan. Außerdem: Wie viele Zustände könnte es für das Rätsel
höchstens geben, wenn man nur `(8 + 1) · (5 + 1) · (3 + 1)` rechnet?

**Erwartete Ausgabe:**
```
raetsel (8, 5, 3) -> 7 Schritte, 16 Zustände entdeckt
Plan: [(8, 0, 0), (3, 5, 0), (3, 2, 3), (6, 2, 0), (6, 0, 2), (1, 5, 2), (1, 4, 3), (4, 4, 0)]
gerade (10, 6, 4) -> nicht lösbar, 10 Zustände angesehen
gross (12, 7, 5) -> 11 Schritte, 24 Zustände entdeckt
grobe Obergrenze für das Rätsel: 216
```

In [ ]:
# dein Code

<details>
<summary>Tipp anzeigen</summary>

`lade` liefert Fassungsvermögen, Start und Ziel. Setze die globalen Namen `GROESSE` und `ZIEL`, dann `suche(start)`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
for name in ['raetsel', 'gerade', 'gross']:
    GROESSE, start_gef, ZIEL = lade('daten/gefaesse_' + name + '.txt')
    plan, entdeckt = suche(start_gef)
    if plan is None:
        print(name, GROESSE, '-> nicht lösbar,', entdeckt, 'Zustände angesehen')
    else:
        print(name, GROESSE, '->', len(plan) - 1, 'Schritte,', entdeckt, 'Zustände entdeckt')
        if name == 'raetsel':
            print('Plan:', plan)
print('grobe Obergrenze für das Rätsel:', (8 + 1) * (5 + 1) * (3 + 1))
```

Das Rätsel geht in 7 Schritten. Die Obergrenze 216 ist viel zu grob: Die Gesamtmenge 8 bleibt immer gleich, also sind
nur Tripel mit Summe 8 möglich, und davon hat die Suche nur 16 entdeckt. Bei `gerade` (10, 6, 4 mit 5 im ersten
Gefäß) ist nichts zu machen: Alle Zahlen sind gerade, und eine ungerade Menge kann nie entstehen. Die Suche merkt das ohne
Rechnung: Sie sieht alle 10 erreichbaren Zustände an und findet keinen mit 5. Solche Fälle ohne Lösung
gibt es in der echten Aufgabe auch.

</details>

## Wettbewerbs-Ecke 2: Hüpfburg (für Schnelle)

*Angelehnt an Aufgabe 5 „Hüpfburg“ der 1. Runde des 41. Bundeswettbewerbs Informatik (2022/23).*

Sasha und Mika laufen über einen Parcours aus Feldern mit Pfeilen. Sasha startet auf Feld 1 und Mika auf Feld 2. In jedem
Schritt springen **beide gleichzeitig**, jeder entlang eines Pfeils von seinem Feld. Der Parcours ist geschafft, sobald
beide auf demselben Feld landen. Kann man das erreichen, und mit welchen Sprüngen?

Die Dateien `daten/huepfburg*.txt` haben in der ersten Zeile die Zahl der Felder und die Zahl der Pfeile, danach je Zeile
einen Pfeil `von nach`. Der Parcours ist ein Graph, aber **kein Knoten ist ein Feld**: Ein Zustand ist das Paar
`(feld_sasha, feld_mika)`. Die Figuren sind unterscheidbar (siehe 3.2), also kommt keine Normalform ins Spiel.

#### Aufgabe E.3

Schreibe `nachfolger(zustand)` für ein Paar von Feldern und `ist_ziel(zustand)`. `lies_burg` ist fertig und gibt zu jedem
Feld die Liste der Felder zurück, auf die ein Pfeil zeigt.

**Erwartete Ausgabe:**
```
20 Felder
[18, 8, 4] [3, 19]
[(18, 3), (18, 19), (8, 3), (8, 19), (4, 3), (4, 19)]
```

In [ ]:
def lies_burg(dateiname):
    zeilen = open(dateiname).read().strip().split('\n')
    anzahl = int(zeilen[0].split()[0])
    pfeile = {}
    for feld in range(1, anzahl + 1):
        pfeile[feld] = []
    for zeile in zeilen[1:]:
        von, nach = zeile.split()
        pfeile[int(von)].append(int(nach))
    return pfeile

def nachfolger(zustand):
    liste = []
    # dein Code
    return liste

def ist_ziel(zustand):
    return False

burg = lies_burg('daten/huepfburg0.txt')
print(len(burg), 'Felder')
print(burg[1], burg[2])
print(nachfolger((1, 2)))

<details>
<summary>Tipp anzeigen</summary>

Zwei verschachtelte Schleifen: erst über die Ziele von Sashas Feld `burg[zustand[0]]`, darin über die Ziele von Mikas Feld
`burg[zustand[1]]`. Jede Kombination ist ein Nachfolger.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def lies_burg(dateiname):
    zeilen = open(dateiname).read().strip().split('\n')
    anzahl = int(zeilen[0].split()[0])
    pfeile = {}
    for feld in range(1, anzahl + 1):
        pfeile[feld] = []
    for zeile in zeilen[1:]:
        von, nach = zeile.split()
        pfeile[int(von)].append(int(nach))
    return pfeile

def nachfolger(zustand):
    liste = []
    for neu_sasha in burg[zustand[0]]:           # jeder Pfeil von Sashas Feld
        for neu_mika in burg[zustand[1]]:        # mit jedem Pfeil von Mikas Feld
            liste.append((neu_sasha, neu_mika))
    return liste

def ist_ziel(zustand):
    return zustand[0] == zustand[1]

burg = lies_burg('daten/huepfburg0.txt')
print(len(burg), 'Felder')
print(burg[1], burg[2])
print(nachfolger((1, 2)))
```

Von `(1, 2)` aus hat Sasha 3 Pfeile und Mika 2, das gibt 6 Nachfolger. Die Zahl der Nachfolger eines Zustands ist das Produkt
der Pfeile der beiden Felder.

</details>

#### Aufgabe E.4

Löse `huepfburg0.txt`, `huepfburg1.txt` und `huepfburg3.txt` mit `suche((1, 2))`. Gib bei einer Lösung die Zahl der Sprünge,
bei kurzen Plänen (höchstens 5 Sprünge) auch die Felder von Sasha und Mika aus, und gib immer die Zahl der entdeckten
Zustände aus. Wie viele Paare gibt es höchstens?

**Erwartete Ausgabe:**
```
huepfburg0 : 3 Sprünge, 63 von 400 Paaren entdeckt
  Sasha: [1, 18, 13, 10]
  Mika:  [2, 19, 20, 10]
huepfburg1 : 121 Sprünge, 272 von 289 Paaren entdeckt
huepfburg3 : nicht lösbar, 72 von 576 Paaren erreichbar
```

In [ ]:
# dein Code

<details>
<summary>Tipp anzeigen</summary>

`burg = lies_burg(...)` setzen (`nachfolger` benutzt dieses `burg`), dann `suche((1, 2))`. Die Felder von Sasha sind die
ersten Einträge der Paare im Plan, die von Mika die zweiten. Höchstens `felder * felder` Paare.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
for nummer in [0, 1, 3]:
    burg = lies_burg('daten/huepfburg' + str(nummer) + '.txt')
    plan, entdeckt = suche((1, 2))
    felder = len(burg)
    if plan is None:
        print('huepfburg' + str(nummer), ': nicht lösbar,', entdeckt, 'von', felder * felder, 'Paaren erreichbar')
    else:
        print('huepfburg' + str(nummer), ':', len(plan) - 1, 'Sprünge,', entdeckt, 'von', felder * felder, 'Paaren entdeckt')
        if len(plan) <= 6:
            print('  Sasha:', [paar[0] for paar in plan])
            print('  Mika: ', [paar[1] for paar in plan])
```

Beim ersten Parcours treffen sich beide nach 3 Sprüngen auf Feld 10. Beim dritten gibt es keine Lösung: Nur 72 der 576 Paare
sind erreichbar, und in keinem stehen beide auf demselben Feld. Beim zweiten Parcours braucht man **121** Sprünge, obwohl er
nur 17 Felder hat. Der Zustandsraum hat höchstens 17 · 17 = 289 Paare, gefunden wurden 272. Von Hand findet man so einen
langen Plan kaum, für die Breitensuche über so wenige Zustände ist er kein Problem.

</details>

#### Frage E.5

1. Wie groß kann der Zustandsraum bei der Hüpfburg höchstens werden, wenn der Parcours `V` Felder und jedes Feld höchstens `d`
   Pfeile hat? Wie lange braucht dann die Breitensuche ungefähr?
2. Was ändert sich, wenn Sasha, Mika **und Lena** gleichzeitig springen?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Es gibt höchstens **V · V** Paare. Von jedem Paar aus gibt es höchstens d · d Nachfolger. Die Breitensuche braucht also
   **O(V² · d²)**. Für 150 Felder (huepfburg2) sind das 22 500 Paare, kein Problem.
2. Der Zustand wird ein Tripel, es gibt **V³** Zustände und d³ Nachfolger je Zustand: O(V³ · d³). Für 150 Felder wären das über
   3 Millionen Tripel, mit einem Parcours aus 1000 Feldern eine Milliarde, zu viel. Je mehr Figuren, desto schneller wächst
   der Zustandsraum. Darum schätzt man seine Größe **vor** dem Programmieren ab.

</details>

## Das hast du gelernt

```python
from collections import deque

# Zustand = Tupel; nachfolger(zustand) und ist_ziel(zustand) gehören zum Rätsel
vorgaenger = {start: None}
schlange = deque([start])
while len(schlange) > 0:
    zustand = schlange.popleft()
    if ist_ziel(zustand):
        break                               # Plan über vorgaenger zurückverfolgen
    for neu in nachfolger(zustand):
        if neu not in vorgaenger:           # beim Einreihen markieren
            vorgaenger[neu] = zustand
            schlange.append(neu)
```

- Ein **Zustand** beschreibt die ganze Lage als **Tupel** (Listen gehen nicht als Schlüssel). Ein Zug ist eine Kante.
- Die Breitensuche aus Notebook 05 bleibt gleich, nur `nachfolger(zustand)` ersetzt die Nachbarliste.
- **Größe des Zustandsraums** abschätzen: Produkt der Möglichkeiten je Angabe (2⁴ bei Wolf, Ziege, Kohl, V² bei zwei
  Figuren), oft sind viel weniger erreichbar. Daraus folgt die Laufzeit.
- **Nicht lösbar** ist sicher, wenn der Zustandsraum endlich ist und die Schlange leer wird, ohne dass das Ziel kam.
  Bei unendlichem Raum braucht man eine Grenze, die keine Lösungen abschneidet.
- **Normalform** (z. B. sortieren) fasst gleichwertige Zustände zusammen. Unterscheidbare Dinge dürfen nicht
  zusammengefasst werden, und ein Zustand muss alles enthalten, was für die Züge wichtig ist.

### Aufgaben der 1. Runde dazu

Bearbeite eine Aufgabe zuerst selbst und sieh dir das Lösungsnotebook erst danach an.

- [Faires Füllen](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf1/33_FairesFuellen/fairesfuellen_loesung.ipynb) (33. BwInf): Gefäße umfüllen, beliebig viele
- [Telepaartie](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf1/38_Telepaartie/telepaartie_loesung.ipynb) (38.): Zustand = Verteilung, Zug = Telepaartie
- [Hüpfburg](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf1/41_Huepfburg/huepfburg_loesung.ipynb) (41.): zwei Figuren gleichzeitig
- [Rotation](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf1/35_Rotation/rotation_loesung.ipynb) (35.): Zustand = Bild des Rahmens, Zug = Drehung

Im nächsten Notebook kosten die Schritte **verschieden viel**. Dann reicht die Breitensuche nicht mehr, und du lernst
**Dijkstra** kennen (Notebook 07).